# Data and Analysis Archiving

This notebook packages all components necessary to fully recapitulate an analysis into a timestamped archive folder. This includes:
- Raw data and metadata (config.yaml files)
- Processed/compiled data
- All notebooks and scripts
- Configuration files (requirements.txt, .gitignore, etc.)
- Python modules (.py files)

The archive is self-contained and can be shared or stored for reproducibility.

## Setup and Configuration

Import required libraries and define the archive destination. Archives are saved to `data/archive/` with timestamps.

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import json

# Project root directory
project_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()

# Archive destination
archive_base = project_root / "data" / "archive"
archive_base.mkdir(parents=True, exist_ok=True)

# Generate timestamp for this archive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_name = f"celegans_analysis_{timestamp}"
archive_dir = archive_base / archive_name

print(f"Project root: {project_root}")
print(f"Archive will be created at: {archive_dir}")

## Define Archive Contents

Specify all files and folders to include in the archive for full reproducibility.

In [ ]:
def gather_archive_contents(root_dir: Path):
    """
    Gather all files and folders needed for a complete archive.
    Returns a dict mapping source paths to relative archive paths.
    """
    archive_contents = {}
    
    # 1. Raw data (all genotype folders with config.yaml and video data)
    raw_data_dir = root_dir / "data" / "raw"
    if raw_data_dir.exists():
        for item in raw_data_dir.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(root_dir)
                archive_contents[item] = rel_path
    
    # 2. Processed/compiled data
    processed_dir = root_dir / "data" / "processed"
    if processed_dir.exists():
        for item in processed_dir.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(root_dir)
                archive_contents[item] = rel_path
    
    # 3. All notebooks
    notebooks_dir = root_dir / "Notebooks"
    if notebooks_dir.exists():
        for nb in notebooks_dir.glob("*.ipynb"):
            rel_path = nb.relative_to(root_dir)
            archive_contents[nb] = rel_path
    
    # 4. All scripts
    scripts_dir = root_dir / "scripts"
    if scripts_dir.exists():
        for script in scripts_dir.rglob("*.py"):
            rel_path = script.relative_to(root_dir)
            archive_contents[script] = rel_path
    
    # 5. Root-level Python files
    for py_file in root_dir.glob("*.py"):
        rel_path = py_file.relative_to(root_dir)
        archive_contents[py_file] = rel_path
    
    # 6. Configuration files
    config_files = [
        "requirements.txt",
        ".gitignore",
        "README.md",
        "setup.py",
        "pyproject.toml",
        "setup.cfg"
    ]
    for cfg in config_files:
        cfg_path = root_dir / cfg
        if cfg_path.exists():
            rel_path = cfg_path.relative_to(root_dir)
            archive_contents[cfg_path] = rel_path
    
    return archive_contents


# Gather all content
contents = gather_archive_contents(project_root)
print(f"Found {len(contents)} files to archive")
print(f"\nBreakdown by directory:")

# Count by top-level directory
from collections import Counter
dir_counts = Counter([str(path).split('/')[0] if '/' in str(path) or '\\' in str(path) else 'root' 
                      for path in contents.values()])
for dir_name, count in sorted(dir_counts.items()):
    print(f"  {dir_name}: {count} files")

## Create Archive and Copy Files

Create the timestamped archive directory and copy all files while preserving the directory structure.

In [ ]:
def create_archive(archive_path: Path, contents: dict, create_manifest: bool = True):
    """
    Copy all files to archive directory, preserving structure.
    
    Args:
        archive_path: Destination archive directory
        contents: Dict mapping source paths to relative archive paths
        create_manifest: Whether to create a JSON manifest of archived files
    
    Returns:
        Tuple of (success_count, total_count)
    """
    archive_path.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    failed_files = []
    manifest_data = {
        "archive_name": archive_path.name,
        "created": datetime.now().isoformat(),
        "total_files": len(contents),
        "files": []
    }
    
    print(f"Creating archive at: {archive_path}")
    print(f"Copying {len(contents)} files...")
    
    for source, rel_dest in contents.items():
        dest = archive_path / rel_dest
        
        try:
            # Create parent directories if needed
            dest.parent.mkdir(parents=True, exist_ok=True)
            
            # Copy file
            shutil.copy2(source, dest)
            success_count += 1
            
            # Add to manifest
            manifest_data["files"].append({
                "path": str(rel_dest),
                "size_bytes": source.stat().st_size,
                "modified": datetime.fromtimestamp(source.stat().st_mtime).isoformat()
            })
            
            # Progress indicator
            if success_count % 10 == 0:
                print(f"  Copied {success_count}/{len(contents)} files...", end='\r')
                
        except Exception as e:
            failed_files.append((source, str(e)))
            print(f"\n  Warning: Failed to copy {source}: {e}")
    
    print(f"\nCompleted: {success_count}/{len(contents)} files copied successfully")
    
    # Save manifest
    if create_manifest:
        manifest_path = archive_path / "MANIFEST.json"
        with open(manifest_path, 'w') as f:
            json.dump(manifest_data, f, indent=2)
        print(f"Manifest saved to: {manifest_path}")
    
    # Report any failures
    if failed_files:
        print(f"\n⚠️  {len(failed_files)} files failed to copy:")
        for path, error in failed_files[:5]:  # Show first 5
            print(f"  - {path}: {error}")
        if len(failed_files) > 5:
            print(f"  ... and {len(failed_files) - 5} more")
    
    return success_count, len(contents)


# Execute archiving
success, total = create_archive(archive_dir, contents, create_manifest=True)
print(f"\n{'='*60}")
print(f"✅ Archive created successfully!")
print(f"   Location: {archive_dir}")
print(f"   Files: {success}/{total}")
print(f"   Size: {sum(f.stat().st_size for f in contents.keys()) / 1024 / 1024:.2f} MB")
print(f"{'='*60}")

## Verify Archive Contents

Display a summary of what was archived and verify the archive structure.

In [ ]:
import pandas as pd

# Load and display manifest
manifest_path = archive_dir / "MANIFEST.json"
if manifest_path.exists():
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    
    print(f"Archive: {manifest['archive_name']}")
    print(f"Created: {manifest['created']}")
    print(f"Total files: {manifest['total_files']}")
    print("\n" + "="*80)
    
    # Create summary DataFrame
    files_df = pd.DataFrame(manifest['files'])
    files_df['size_kb'] = files_df['size_bytes'] / 1024
    files_df['directory'] = files_df['path'].apply(lambda x: str(Path(x).parts[0]) if len(Path(x).parts) > 0 else 'root')
    
    # Summary by directory
    print("\nArchive contents by directory:")
    summary = files_df.groupby('directory').agg({
        'path': 'count',
        'size_kb': 'sum'
    }).rename(columns={'path': 'file_count', 'size_kb': 'total_size_kb'})
    summary['total_size_mb'] = summary['total_size_kb'] / 1024
    summary = summary.sort_values('total_size_mb', ascending=False)
    
    display(summary[['file_count', 'total_size_mb']].round(2))
    
    print(f"\n✅ Archive is ready for distribution or long-term storage")
    print(f"   Path: {archive_dir}")
else:
    print("Warning: Manifest file not found!")

## Notes

**What's included in the archive:**
- All raw data from `data/raw/` (including config.yaml metadata files)
- All processed/compiled data from `data/processed/`
- All Jupyter notebooks from `Notebooks/`
- All Python scripts from `scripts/` and root directory
- Configuration files: requirements.txt, .gitignore, README.md, etc.

**Archive structure:**
- Archives are saved to `data/archive/` with timestamp: `celegans_analysis_YYYYMMDD_HHMMSS/`
- A `MANIFEST.json` file is included with metadata about all archived files
- Directory structure is preserved for easy navigation

**Use cases:**
- **Long-term storage**: Archive completed analyses with all necessary files
- **Sharing**: Send complete, reproducible analysis packages to collaborators
- **Backup**: Create snapshots before major changes to the analysis
- **Publication**: Package analysis for supplementary materials

**To restore an archive:**
Simply extract the archive contents to recreate the full analysis environment, install dependencies from requirements.txt, and re-run the notebooks.